In [9]:
import requests
import json
import pandas as pd

# Optional: Adjust display options for clean Jupyter notebook printing
pd.set_option('display.max_columns', 15)
pd.set_option('display.width', 1000)

def fetch_tradingview_swing_data():
    """
    Fetches real-time market data and daily indicators (RSI, MACD, 52W range, RVOL)
    directly from TradingView's India screener API in one batch.
    """
    url = "https://scanner.tradingview.com/india/scan"
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/128.0.0.0 Safari/537.36",
        "Content-Type": "application/json"
    }

    # Query columns matching daily indicators and swing filters
    columns = [
        "name",                       # 0. Ticker Symbol
        "description",                # 1. Company Name
        "close",                      # 2. LTP
        "change",                     # 3. % Change Today
        "price_52_week_high",         # 4. 52-Week High
        "price_52_week_low",          # 5. 52-Week Low
        "RSI",                        # 6. Daily RSI(14)
        "MACD.macd",                  # 7. Daily MACD Line
        "MACD.signal",                # 8. Daily MACD Signal Line
        "relative_volume_10d_calc",   # 9. RVOL (Relative Volume vs 10D avg)
        "volume",                     # 10. Day Volume
        "sector"                      # 11. Sector classification
    ]

    payload = {
        "filter": [
            {"left": "exchange", "operation": "equal", "right": "NSE"},
            {"left": "type", "operation": "equal", "right": "stock"},
            {"left": "is_primary", "operation": "equal", "right": True},
            {"left": "active_symbol", "operation": "equal", "right": True}
        ],
        "options": {"lang": "en"},
        "symbols": {"query": {"types": []}},
        "columns": columns,
        "sort": {"sortBy": "change", "sortOrder": "desc"},
        "range": [0, 300]  # Scans top 300 active NSE equities
    }

    response = requests.post(url, headers=headers, json=payload, timeout=12)
    response.raise_for_status()
    raw_data = response.json().get("data", [])

    records = []
    for item in raw_data:
        vals = item.get("d", [])
        if len(vals) < len(columns):
            continue

        close_price = vals[2] or 0.0
        high_52w = vals[4] or 1.0
        low_52w = vals[5] or 1.0

        # Proximity: % distance to 52-week high (100% means right at 52W High)
        proximity_52w = round((close_price / high_52w) * 100, 2) if high_52w else 0.0

        records.append({
            "symbol": vals[0],
            "company": vals[1],
            "ltp": round(close_price, 2),
            "pchange": round(vals[3] or 0.0, 2),
            "52w_high": round(high_52w, 2),
            "52w_low": round(low_52w, 2),
            "proximity_52w_%": proximity_52w,
            "rsi": round(vals[6] or 50.0, 1),
            "macd": round(vals[7] or 0.0, 2),
            "macd_signal": round(vals[8] or 0.0, 2),
            "rvol": round(vals[9] or 1.0, 2),
            "volume": vals[10] or 0,
            "sector": vals[11] or "Others"
        })

    return pd.DataFrame(records)


def compute_market_flow(df: pd.DataFrame):
    """
    Computes top-down metrics:
    1. Overall Market Breadth & Advance/Decline Bias
    2. Sector Relative Strength (RS)
    3. Composite Flow Score for swing entries
    """
    # 1. Market Breadth
    advances = (df["pchange"] > 0).sum()
    declines = (df["pchange"] < 0).sum()
    ad_ratio = round(advances / max(1, declines), 2)

    if ad_ratio >= 1.75:
        market_bias = "STRONG BULLISH (Aggressive Long Momentum)"
    elif ad_ratio >= 1.15:
        market_bias = "MILD BULLISH (Selective Buying)"
    elif ad_ratio <= 0.6:
        market_bias = "STRONG BEARISH (Cash / Capital Preservation)"
    elif ad_ratio <= 0.85:
        market_bias = "MILD BEARISH (Caution / Rangebound)"
    else:
        market_bias = "NEUTRAL / CHOPPY"

    # 2. Sector Strength Analysis
    sector_summary = (
        df[df["sector"] != "Others"]
        .groupby("sector")
        .agg(
            avg_change=("pchange", "mean"),
            advances=("pchange", lambda x: (x > 0).sum()),
            count=("pchange", "count")
        )
        .reset_index()
    )
    sector_summary["breadth_%"] = round((sector_summary["advances"] / sector_summary["count"]) * 100, 1)
    sector_summary["avg_change"] = sector_summary["avg_change"].round(2)
    sector_summary = sector_summary.sort_values(by="avg_change", ascending=False)

    top_sectors = set(sector_summary.head(3)["sector"].tolist())

    # 3. Composite Flow Score Calculation
    def calculate_score(row):
        score = 0.0

        # A. 52W Proximity (Max 35 pts) - rewards stocks near breakout
        prox = row["proximity_52w_%"]
        if prox >= 95.0:
            score += 35.0
        elif prox >= 85.0:
            score += 25.0
        elif prox >= 75.0:
            score += 15.0

        # B. RSI Swing Sweet Spot 55-68 (Max 30 pts)
        rsi = row["rsi"]
        if 55.0 <= rsi <= 68.0:
            score += 30.0
        elif 50.0 <= rsi < 55.0 or 68.0 < rsi <= 75.0:
            score += 18.0
        elif rsi > 75.0:
            score += 8.0  # Overbought exhaustion penalty

        # C. Volume Confirmation (Max 20 pts)
        rvol = row["rvol"]
        if rvol >= 1.5:
            score += 20.0
        elif rvol >= 1.1:
            score += 12.0

        # D. Sector Alignment Bonus (15 pts)
        if row["sector"] in top_sectors:
            score += 15.0

        return round(score, 1)

    df["flow_score"] = df.apply(calculate_score, axis=1)

    return market_bias, ad_ratio, advances, declines, sector_summary, df


# ================= EXECUTION =================
if __name__ == "__main__":
    print("Connecting to TradingView India Scanner...\n")
    df_raw = fetch_tradingview_swing_data()
    bias, ad_ratio, adv, dec, sectors, scored_df = compute_market_flow(df_raw)

    print("=" * 80)
    print(f" MARKET REGIME: {bias}")
    print(f" Advance/Decline Ratio: {ad_ratio} | Advances: {adv} | Declines: {dec}")
    print("=" * 80)

    print("\n--- SECTOR MOMENTUM (TOP LEADING SECTORS) ---")
    print(sectors.head(6).to_string(index=False))

    print("\n" + "=" * 80)
    print(" TOP 10 SWING CANDIDATES (SORTED BY COMPOSITE FLOW SCORE)")
    print("=" * 80)
    
    # Select stocks with positive price action, sorted by Flow Score
    swing_leads = (
        scored_df[scored_df["pchange"] > 0]
        .sort_values(by=["flow_score", "pchange"], ascending=[False, False])
        [['symbol', 'ltp', 'pchange', 'proximity_52w_%', 'rsi', 'macd', 'rvol', 'sector', 'flow_score']]
        .head(10)
    )
    print(swing_leads.to_string(index=False))

Connecting to TradingView India Scanner...

 MARKET REGIME: STRONG BULLISH (Aggressive Long Momentum)
 Advance/Decline Ratio: 300.0 | Advances: 300 | Declines: 0

--- SECTOR MOMENTUM (TOP LEADING SECTORS) ---
               sector  avg_change  advances  count  breadth_%
      Health Services        6.52         1      1      100.0
  Commercial Services        5.79        11     11      100.0
    Consumer Durables        5.57        20     20      100.0
Electronic Technology        5.43         7      7      100.0
   Process Industries        5.10        53     53      100.0
Distribution Services        4.57        13     13      100.0

 TOP 10 SWING CANDIDATES (SORTED BY COMPOSITE FLOW SCORE)
    symbol    ltp  pchange  proximity_52w_%  rsi  macd  rvol                 sector  flow_score
  RBZJEWEL 209.28     4.95            99.95 64.6  7.62  3.54      Consumer Durables       100.0
  SHANKESH 101.06     3.48            87.20 59.5  0.84  2.77      Consumer Durables        90.0
  SWARNSAR

In [ ]:
https://gemini.google.com/app/40ae676fc5f7d6a0